In [ ]:
import os, copy
import glob

import numpy as np
import scipy.optimize as so
import pandas as pd
import xarray as xr
import rioxarray as rxr

import netCDF4
import h5py
from osgeo import gdal

%matplotlib inline  
import matplotlib as mpl
import matplotlib.pyplot as plt
import colorcet as cc

import panel as pn
pn.extension()
opj = os.path.join


In [ ]:
workdir = '/sat_data/satellite/acix-iii/results'
workdir='/media/harmel/TOSHIBA EXT/acix-iii'
#workdir='/media/harmel/TOSHIBA EXT/data/satellite/prisma/zoffoli/L2A'
#workdir='/DATA/git/satellite_app/hgrs/'
files = pn.widgets.FileSelector(workdir)

files


In [ ]:
img =xr.open_dataset(files.value[0])
import datetime as dt
date = dt.datetime.strptime(img.acquisition_date,'%Y-%m-%dT%H:%M:%S.%f')
#img.acquisition_date

In [ ]:
img.vza

In [ ]:
Ttot_Ed=xr.open_dataset('/DATA/git/satellite_app/hgrs/data/lut/transmittance_downward_irradiance.nc')
sza=np.nanmean(img.sza)
vza= np.nanmean(img.vza)
aot_ref= np.nanmean(img.aot_ref)
model = img.aerosol_model
wl = img.wl

In [ ]:
Ttot_Ed_ = Ttot_Ed.Ttot_Ed.sel(model=model).interp(sza=sza, method='cubic').interp(aot_ref=aot_ref, method='quadratic').interp(wl=wl, method='cubic')
Ttot_Lu_ = Ttot_Ed.Ttot_Ed.sel(model=model).interp(sza=vza, method='cubic').interp(aot_ref=aot_ref, method='quadratic').interp(wl=wl, method='cubic')**1.05
Ttot = (Ttot_Ed_ *Ttot_Lu_).reset_coords(drop=True)
Ttot

In [ ]:
Ttot_Ed=xr.open_dataset('/DATA/git/satellite_app/hgrs/data/lut/transmittance_downward_irradiance.nc')
cmap = mpl.colors.LinearSegmentedColormap.from_list("",
                                                    ['navy', "blue", 'lightskyblue',
                                                     "grey",   'forestgreen','yellowgreen',
                                                     "khaki", "gold",
                                                     'orangered', "firebrick", 'purple'])

norm = mpl.colors.Normalize(vmin=0.001,vmax=1.1)#-3, vmax=np.log10(1.5))
rh='_rh70'
fig, axs = plt.subplots(3, 3, figsize=(18, 12), sharey=True, sharex=True)
axs = axs.ravel()
for i_, model_ in enumerate(Ttot_Ed.model.values):
    model = model_ #+ rh
    Ttot_Ed_ = Ttot_Ed.Ttot_Ed.sel(model=model).interp(sza=sza, method='cubic').interp(wl=wl, method='cubic')
    Ttot_Lu_ = Ttot_Ed.Ttot_Ed.sel(model=model).interp(sza=vza, method='cubic').interp(wl=wl, method='cubic')**1.05
    Ttot = Ttot_Ed_ *Ttot_Lu_
   
    for aot_ref in Ttot_Ed.aot_ref.values[1:]:
        Ttot.sel(aot_ref=aot_ref).plot(x='wl', color=cmap(norm(aot_ref)),label=str(aot_ref),  lw=1, ax=axs[i_])
        
    axs[i_].set_title(model)
    axs[i_].minorticks_on()
    axs[i_].legend(title='aot(550)')
plt.tight_layout()

## Plot and interact

In [ ]:
Ttot #.reset_coords()

In [ ]:

param = 'Rrs' #Rtoa'
#img = prod[['Rtoa','Ltoa']] 
img['Rrs_corr'] = img[param]/Ttot


In [ ]:
from holoviews import streams
import holoviews as hv
import panel as pn
import param
import numpy as np
import xarray as xr
hv.extension('bokeh')
from holoviews import opts


opts.defaults(
    opts.GridSpace(shared_xaxis=True, shared_yaxis=True),
    opts.Image(cmap='binary_r', width=800, height=700),
    opts.Labels(text_color='white', text_font_size='8pt', text_align='left', text_baseline='bottom'),
    opts.Path(color='white'),
    opts.Spread(width=900),
    opts.Overlay(show_legend=True))
# set the parameter for spectra extraction
hv.extension('bokeh')
pn.extension()


param = 'Rrs_corr' #Rtoa'
#img = prod[['Rtoa','Ltoa']] 
raster = img[param]#L2grs #masked[param] 

#img = prod[['Rtoa','Ltoa']] 
vmax = 0.03
#param = 'rho'
#raster = dc_l2c[param] 
cmap='RdBu_r'
cmap='Spectral_r'
third_dim = 'wl'

wl= raster.wl.data
Nwl = len(wl)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap=cmap ,colorbar=True,clim=(0.00,vmax)).hist(bin_range=(0,0.2)) 

polys = hv.Polygons([])
box_stream = hv.streams.BoxEdit(source=polys)
dmap, dmap_std=[],[]

def roi_curves(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]= hv.Curve((wl,mean[param]),'Wavelength (nm)', param) 

    return hv.NdOverlay(curves)


# a bit dirty to have two similar function, but holoviews does not like mixing Curve and Spread for the same stream
def roi_spreads(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]=  hv.Spread((wl,mean[param],std[param]),fill_alpha=0.3)

    return hv.NdOverlay(curves)

mean=hv.DynamicMap(roi_curves,streams=[box_stream])
std =hv.DynamicMap(roi_spreads, streams=[box_stream])    
hlines = hv.HoloMap({wl[i]: hv.VLine(wl[i]) for i in range(Nwl)},third_dim )

widget = pn.widgets.RangeSlider(start=0, end=vmax,step=0.001)

jscode = """
    color_mapper.low = cb_obj.value[0];
    color_mapper.high = cb_obj.value[1];
"""
link = widget.jslink(im, code={'value': jscode})

hv.output(widget_location='top_left')

# visualize and play
graphs = ((mean* std *hlines).relabel(param))
layout = (im * polys +graphs    ).opts(opts.Image(tools=['hover']),
    opts.Curve(width=750,height=500, framewise=True,xlim=(400,1140),tools=['hover']), 
    opts.Polygons(fill_alpha=0.2, color='green',line_color='black'), 
    opts.VLine(color='black')).cols(2)
layout 

In [ ]:
params=['aot_ref','aot_ref_std','tcwv','tcwv_std','brdfg','brdfg_std']

fig,axs = plt.subplots(3,2,figsize=(12,15))
axs=axs.ravel()

for i in range(len(params)):
    img[params[i]].plot.imshow(cmap=plt.cm.Spectral_r, robust=True,vmin=0,#vmax=0.201,
                               cbar_kwargs={'shrink': 0.78,'label':params[i]},ax=axs[i]) # extent=extent_val, transform=proj, 
    axs[i].set(xticks=[], yticks=[])
    axs[i].set_ylabel('')
    axs[i].set_xlabel('')    
    axs[i].set_title(params[i])    

In [ ]:
params=['aot_ref_full','tcwv_full','brdfg_full']

fig,axs = plt.subplots(1,3,figsize=(18,4))
axs=axs.ravel()

for i in range(len(params)):
    img[params[i]].plot.imshow(cmap=plt.cm.Spectral_r, robust=True,vmin=0,#vmax=0.201,
                               cbar_kwargs={'shrink': 0.78,'label':params[i]},ax=axs[i]) # extent=extent_val, transform=proj, 
    axs[i].set(xticks=[], yticks=[])
    axs[i].set_ylabel('')
    axs[i].set_xlabel('')    
    axs[i].set_title(params[i])    

In [ ]:
gamma=1#.5

fig,axs = plt.subplots(1,2,figsize=(12,6))
axs=axs.ravel()

rgb=img.Rrs.isel(wl=[30,20,6])
adj = xr.DataArray([1.2,1,2],coords={"wl":rgb.wl})
fig = ((rgb*adj)**gamma).plot.imshow(rgb='wl',robust=True,ax=axs[0])#, subplot_kws=dict(projection= l1c.proj))
img['brdfg_full'].plot.imshow(cmap=plt.cm.gray, robust=True,vmin=0,ax=axs[1],add_colorbar=False)

for i in range(2):
    axs[i].set(xticks=[], yticks=[])
    axs[i].set_ylabel('')
    axs[i].set_xlabel('')    
plt.tight_layout()        

In [ ]:
model='COAV_rh70'
Ttot_Ed_ = Ttot_Ed.Ttot_Ed.sel(model=model).interp(sza=sza, method='cubic').interp(wl=wl, method='cubic')
Ttot_Lu_ = Ttot_Ed.Ttot_Ed.sel(model=model).interp(sza=vza, method='cubic').interp(wl=wl, method='cubic')**1.05
Ttot = Ttot_Ed_ *Ttot_Lu_
Ttot

In [ ]:
Ttot_img = Ttot.interp(aot_ref=img.aot_ref_full)

In [ ]:
Rrs_boa = img.Rrs/Ttot_img

In [ ]:
Rrs_boa

In [ ]:

from aeronet_visu import data_loading as dl
opj = os.path.join
dir = '/DATA/AERONET/OCv3/'
figdir= '/DATA/AERONET/fig'

aeronet_site = 'South_Greenbay'
aeronet_site = 'Galata_Platform'
aeronet_site = 'Lake_Erie'
aeronet_site = 'Venise'
aeronet_site = 'Bahia_Blanca'
file = aeronet_site+'_OCv3.lev15'

irr = dl.irradiance()
irr.load_F0()

params = ['Lwn','Lwn_IOP','Lwn_f/Q']

# ---------------------------------------------
# Load data and convert into xarray
# ---------------------------------------------

df = dl.read(opj(dir, file)).read_aeronet_ocv3()

df = df.droplevel(0, 1)
# criteria to select scalar and spectrum values
criteria = df.columns.get_level_values(1) == ''
df_att = df.loc[:, criteria].droplevel(1, 1)
df_spec = df.loc[:, ~criteria]

ds = df_spec.stack().to_xarray()
ds = xr.merge([ds, df_att.to_xarray()])
del df

ds = ds.assign_coords({'level_1': ds.level_1.astype(float)}).rename({'level_1': "wl"})
ds['SZA'] = ds.Solar_Zenith_Angle.mean(axis=1)
ds['year']=ds['date.year']
ds['season']=ds['date.season']

wl = ds.wavelength * 1000
for param in params:
    ds['Rrs_'+param] = ds[param] / (irr.get_F0(wl) * 0.1)
ds=ds.sortby("wl")

In [ ]:
ds

In [ ]:

plt.figure(figsize=(8,5))
xcenter,ycenter=500,500
for i in range(6):
    for j in range(6):
        img.Rrs.isel(x=xcenter+i,y=ycenter+j).plot(x='wl',color='grey',alpha=0.5,lw=0.7)#,label='PRISMA')
        img.Rrs_corr.isel(x=xcenter+i,y=ycenter+j).plot(x='wl',color='black',alpha=0.5,lw=0.7)#,label='PRISMA')
for param in params:
    ds['Rrs_'+param].sel(date=str(date),method='nearest').dropna('wl').plot(x='wl',marker='o',label=param)
plt.hlines(0,380,1120,ls=':',lw=0.5,color='black',zorder=0)
plt.ylabel('$R_{rs}\ (sr^{-1})$')
plt.legend()
#plt.xlim(390,1100)

In [ ]:
hours=2
delta = dt.timedelta(hours=hours/2)


In [ ]:
ds['Rrs_Lwn'].sel(date=slice(date-delta,date+delta)).dropna('wl').plot(x='wl',marker='o',hue='date',lw=0.7)

In [ ]:
ds['Aerosol_Optical_Depth'].sel(date=slice(date-delta,date+delta)).dropna('wl').plot(x='wl',marker='o',hue='date',lw=0.7)

In [ ]:
from holoviews import streams
import holoviews as hv
import panel as pn
import param
import numpy as np
import xarray as xr
hv.extension('bokeh')
from holoviews import opts

opts.defaults(
    opts.GridSpace(shared_xaxis=True, shared_yaxis=True),
    opts.Image(cmap='binary_r', width=800, height=700),
    opts.Labels(text_color='white', text_font_size='8pt', text_align='left', text_baseline='bottom'),
    opts.Path(color='white'),
    opts.Spread(width=900),
    opts.Overlay(show_legend=True))
# set the parameter for spectra extraction
hv.extension('bokeh')
pn.extension()

raster = img.Rrs#.reset_coords()#.isel(time=-1,drop=True)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap= 'RdBu_r',colorbar=True)#.hist(bin_range=(0,0.02) ) 
widget = pn.widgets.RangeSlider(start=0, end=1,step=0.001)

jscode = """
    color_mapper.low = cb_obj.value[0];
    color_mapper.high = cb_obj.value[1];
"""
link = widget.jslink(im, code={'value': jscode})
pn.Column(widget, im)